In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('AnimeRecommender') \
    .config('spark.ui.port', '4040') \
    .getOrCreate()

print(f'Spark version: {spark.version}')
print('Spark UI: http://localhost:4040')

Spark version: 3.5.0
Spark UI: http://localhost:4040


In [2]:
BASE = '/home/jovyan/work/data/preprocessed'

reviews_data = spark.read.csv(f'{BASE}/reviews.csv', header=True, inferSchema=True)
anime_data   = spark.read.csv(f'{BASE}/animes.csv',    header=True, inferSchema=True)

In [3]:
from pyspark.ml.feature import StringIndexer
review_indexed = StringIndexer(inputCol="profile", outputCol="user_id")
reviews_data_indexed = review_indexed.fit(reviews_data).transform(reviews_data)

final_data = reviews_data_indexed.select(
    col('user_id').cast('integer'),
    col('anime_id').cast('integer'),
    col('score').cast('float')
).filter(col('score') > 0).filter( col('score') <= 10)

In [4]:
from recommenders import train_item_item, get_similar_items_for_user
from pyspark.ml.recommendation import ALSModel

ui_model = ALSModel.load('/home/jovyan/work/models/user-item')
ui_model_better = ALSModel.load('/home/jovyan/work/models/user-item-better')

trained_ii_model, score_history = train_item_item(anime_data)
target_uid = final_data.select('user_id').first()[0]
def ii_model(user_id, n=10):
    return get_similar_items_for_user(user_id, final_data, trained_ii_model, score_history, n)

In [5]:
from recommenders import hybridV1
from coverage import precision_recall_at_k

#recs = hybridV1(target_uid, ui_model, ii_model, final_data, score_history, spark, n=10)
#print(precision_recall_at_k(target_uid, recs, final_data, k=10, threshold=7.0))

In [6]:
from coverage import catalog_coverage
# Sample users and collect recommendations
sample_users = final_data.select('user_id').distinct().limit(10).toPandas()['user_id'].tolist()

#all_recs = []
#for uid in sample_users:
#    recs = hybridV1(uid, ui_model, ii_model, final_data, score_history, spark, n=10)
#    if recs is not None:
#        all_recs.append(recs)

In [7]:
#catalog_coverage(all_recs, final_data)

In [8]:
from coverage import diversity_score
#print(f'Diversity: {diversity_score(recs, anime_data):.4f}')

In [9]:
from coverage import novelty_score
#print(f'Novelty: {novelty_score(recs, final_data):.4f}')

In [10]:
from coverage import evaluate_all

evaluate_all(target_uid, ui_model, ui_model_better, ii_model, final_data, score_history, anime_data, spark)

Dynamic weights: 0.8
Number of ratings: 109


,precision@k,diversity,novelty
ALS,0.0,0.8800,14.7565
ALS_better,0.2,0.9103,12.5316
Content,0.0,0.2267,12.6573
Hybrid,0.0,0.3083,10.5675
Hybrid (better ALS),0.3,0.8091,11.9076
Hybrid (w=3 ii),0.0,0.7563,13.0742
Hybrid (better ALS) (w=3 ii),0.3,0.8318,12.1398
Hybrid (w=7 ii),0.0,0.2132,9.8774
Hybrid (better ALS) (w=7 ii),0.0,0.4655,11.1121
Hybrid (dynamic),0.0,0.8366,13.8243


In [11]:
target_user = final_data.groupBy("user_id").count().filter(col("count") < 20).first()[0]
evaluate_all(target_user, ui_model, ui_model_better, ii_model, final_data, score_history, anime_data, spark)

Dynamic weights: 0.2
Number of ratings: 4


,precision@k,diversity,novelty
ALS,0.0,0.8655,12.7546
ALS_better,0.0,0.9389,13.7572
Content,0.0,0.1344,13.7904
Hybrid,0.0,0.2504,12.6065
Hybrid (better ALS),0.0,0.4290,12.8713
Hybrid (w=3 ii),0.0,0.7053,13.2842
Hybrid (better ALS) (w=3 ii),0.0,0.8971,14.0379
Hybrid (w=7 ii),0.0,0.0000,11.9806
Hybrid (better ALS) (w=7 ii),0.0,0.0000,11.9075
Hybrid (dynamic),0.0,0.0000,11.9806
